# Name: Nitin Jain Roll No.: M24AID022
# CSL7110 Assignment 4 – Machine Learning for  Big Data
## Clustering, Inverted Index, and PageRank
### Github Repository Link : https://github.com/nitinjain2015/M24AID022_CSL7110_Assignment4

In [2]:
# Cell 1: Importing required Python libraries
import numpy as np
import random
import time
import os
import requests

# Set random seed for consistent output
random.seed(42)
np.random.seed(42)

## Part 1 – Clustering Algorithms (Farthest First and KMeans++)

In [4]:
import os
os.getcwd()
os.listdir()


['.ipynb_checkpoints',
 'actions.txt',
 'answers.txt',
 'Assignment 4- datasets',
 'Assignment 4- datasets.zip',
 'M24AID022_CSL7110_Assignment4.ipynb',
 'Project_Assig4_MLwBData_V01.ipynb',
 'small.txt',
 'spambase.data',
 'webpages',
 'whole.txt']

In [5]:
import numpy as np

def readVectorsSeq(filename):
    P = []
    with open(filename, 'r') as f:
        for line in f:
            nums = [float(x) for x in line.strip().split(',')]
            P.append(np.array(nums))
    return P

In [6]:
vectors = readVectorsSeq("spambase.data")

print(len(vectors))
print(vectors[0])


4601
[  0.      0.64    0.64    0.      0.32    0.      0.      0.      0.
   0.      0.      0.64    0.      0.      0.      0.32    0.      1.29
   1.93    0.      0.96    0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.      0.      0.
   0.      0.      0.      0.      0.      0.      0.778   0.      0.
   3.756  61.    278.      1.   ]


In [7]:
# Cell 4: Squared Euclidean distance
def sqdist(a,b):
    return np.sum((a-b)**2)

In [8]:
# Cell 5: Farthest First Traversal
def kcenter(P,k):
    centers=[random.choice(P)]
    for _ in range(1,k):
        farthest=None
        max_dist=-1
        for p in P:
            d=min(sqdist(p,c) for c in centers)
            if d>max_dist:
                max_dist=d
                farthest=p
        centers.append(farthest)
    return centers

In [9]:
# Cell 6: KMeans++ initialization
def kmeansPP(P,k):
    centers=[random.choice(P)]
    for _ in range(1,k):
        distances=[min(sqdist(p,c) for c in centers) for p in P]
        total=sum(distances)
        probs=[d/total for d in distances]
        cumulative=np.cumsum(probs)
        r=random.random()
        for i,p in enumerate(P):
            if r<=cumulative[i]:
                centers.append(p)
                break
    return centers

In [10]:
# Cell 7: KMeans objective function
def kmeansObj(P,C):
    total = 0
    for p in P:
        d=min(sqdist(p,c) for c in C)
        total+=d
    return total/len(P)

In [11]:
# Cell 8: Running the clustering 
# P = readVectorsSeq('spam_data.txt')

filename = "spambase.data"

P = readVectorsSeq(filename)

print("Dataset size:", len(P))
print("Feature dimension:", len(P[0]))

k = 10
k1 = 50

print("k =", k)
print("k1 =", k1)

start=time.time()
C = kcenter(P,k)
print('kcenter runtime:', time.time()-start)

C = kmeansPP(P,k)
print('kmeans++ objective:', kmeansObj(P,C))

X = kcenter(P,k1)
C = kmeansPP(X,k)
print('Coreset objective:', kmeansObj(P,C))

Dataset size: 4601
Feature dimension: 58
k = 10
k1 = 50
kcenter runtime: 2.2649495601654053
kmeans++ objective: 25429.791217440463
Coreset objective: 78592.54910255819


## Part 2 – Web Search Engine (Inverted Index)

In [13]:
# ==========================================
# Q2: Inverted Index Search Engine
# ==========================================

import re

# Position Class 
class Position:

    def __init__(self, page, index):
        self.page = page
        self.index = index

    def getPageEntry(self):
        return self.page

    def getWordIndex(self):
        return self.index

#  WordEntry Class 
class WordEntry:

    def __init__(self, word):
        self.word = word
        self.positions = []

    def addPosition(self, position):
        self.positions.append(position)

    def getAllPositionsForThisWord(self):
        return self.positions

# PageIndex Class
class PageIndex:

    def __init__(self):
        self.wordEntries = {}

    def addPositionForWord(self, word, position):

        if word not in self.wordEntries:
            self.wordEntries[word] = WordEntry(word)

        self.wordEntries[word].addPosition(position)

# PageEntry Class
class PageEntry:

    def __init__(self, filename):

        self.name = filename
        self.pageIndex = PageIndex()

        with open("webpages/" + filename, "r", encoding="utf-8") as f:

            words = re.findall(r'\w+', f.read().lower())

            for i, word in enumerate(words):

                pos = Position(self, i)
                self.pageIndex.addPositionForWord(word, pos)

# InvertedPageIndex Class
class InvertedPageIndex:

    def __init__(self):
        self.index = {}

    def addPage(self, page):

        for word, entry in page.pageIndex.wordEntries.items():

            if word not in self.index:
                self.index[word] = []

            self.index[word].extend(entry.getAllPositionsForThisWord())

    def getPagesWhichContainWord(self, word):

        if word not in self.index:
            return []

        return sorted(set([p.getPageEntry().name for p in self.index[word]]))

# ==========================================
# Build index using actions.txt
# ==========================================

engine = InvertedPageIndex()

with open("actions.txt") as f:

    for line in f:

        parts = line.strip().split()

        if parts[0] == "addPage":

            page = PageEntry(parts[1])
            engine.addPage(page)

        elif parts[0] == "queryFindPagesWhichContainWord":

            word = re.findall(r'\w+', parts[1].lower())[0]

            result = engine.getPagesWhichContainWord(word)

            if result:
                print(", ".join(result))
            else:
                print("No webpage contains word", word)
                
        elif parts[0] == "queryFindPositionsOfWordInAPage":

            word = re.findall(r'\w+', parts[1].lower())[0]
            page = parts[2]
            
            result = []
            
            if word in engine.index:
                for entry in engine.index[word]:
                    if entry.page == page:
                        result = entry.positions
                        break            

            if result:
                print(", ".join(map(str,result)))
            else:
                print("Webpage", page, "does not contain word", word)

        elif parts[0] == "queryFindPositionsOfWordInAPage":

            word = re.findall(r'\w+', parts[1].lower())[0]
            page = parts[2]

            result = engine.getPositions(word, page)

            if result:
                print(", ".join(map(str,result)))
            else:
                print("Webpage", page, "does not contain word", word)


No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine


## Part 3 – PageRank using Spark

In [15]:
# =====================================
# Q3: PageRank Implementation (Alternate code using Python beacuse of Spark compatabilty issue with Python v 3.12 in local environment)
# =====================================
# Given 
beta = 0.8
iterations = 40


def load_graph(file):

    edges = []
    
    with open(file) as f:
        for line in f:
            u, v = map(int, line.strip().split())
            edges.append((u, v))
    
    return edges


def pagerank(file):

    edges = load_graph(file)

    nodes = set()
    for u, v in edges:
        nodes.add(u)
        nodes.add(v)

    n = len(nodes)

    # outgoing links
    outlinks = {node: [] for node in nodes}

    # incoming links
    inlinks = {node: [] for node in nodes}

    for u, v in edges:
        outlinks[u].append(v)
        inlinks[v].append(u)

    # initial ranks
    ranks = {node: 1/n for node in nodes}

    for _ in range(iterations):

        new_ranks = {}

        for node in nodes:

            rank_sum = 0

            for src in inlinks[node]:
                rank_sum += ranks[src] / len(outlinks[src])

            new_ranks[node] = (1 - beta) / n + beta * rank_sum

        ranks = new_ranks

    return ranks


# =====================================
# Run on SMALL graph (validation)
# =====================================

ranks = pagerank("small.txt")

print("\n==============================")
print("Graph: small.txt")
print("==============================")

sorted_ranks = sorted(ranks.items(), key=lambda x: -x[1])

print("\nTop 5 nodes:")
for node, score in sorted_ranks[:5]:
    print(node, round(score,6))

print("\nBottom 5 nodes:")
for node, score in sorted(ranks.items(), key=lambda x: x[1])[:5]:
    print(node, round(score,6))


# =====================================
# Run on WHOLE graph
# =====================================

ranks = pagerank("whole.txt")

print("\n==============================")
print("Graph: whole.txt")
print("==============================")

sorted_ranks = sorted(ranks.items(), key=lambda x: -x[1])

print("\nTop 5 nodes:")
for node, score in sorted_ranks[:5]:
    print(node, round(score,6))

print("\nBottom 5 nodes:")
for node, score in sorted(ranks.items(), key=lambda x: x[1])[:5]:
    print(node, round(score,6))


Graph: small.txt

Top 5 nodes:
53 0.037869
14 0.035867
1 0.035141
40 0.033831
27 0.03313

Bottom 5 nodes:
85 0.003235
59 0.003444
81 0.00358
37 0.003714
89 0.00384

Graph: whole.txt

Top 5 nodes:
263 0.002017
537 0.001944
965 0.001894
243 0.00185
187 0.00183

Bottom 5 nodes:
558 0.00033
93 0.000351
424 0.000355
62 0.000361
408 0.000388
